# InfraPulse — Dataset Builder & Model Trainer
**Team: Shauryas | Takneek'26**

This notebook does everything for the AI part of InfraPulse:

1. Downloads public defect image datasets
2. Sorts them into our 4 classes: `spalling`, `stagnant_water`, `cracked_tiles`, `paint_peeling`
3. Fine-tunes an ImageNet-pretrained ResNet18 (allowed by the PS — generic backbone only)
4. Evaluates it (confusion matrix, precision, recall, F1)
5. Exports one model file you download and drop into the web app

**Before you start:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save.

Run the cells top to bottom. Don't skip any.

---
### PS compliance note
We start from `ResNet18` pretrained on **ImageNet** (a general-purpose dataset — explicitly permitted).
We do **not** use any checkpoint pretrained on defect/crack/building-damage data. All defect-specific
learning happens here, in our own training run.

## Cell 1 — Check the GPU and install packages

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "NO GPU FOUND")

# If it says NO GPU FOUND: Runtime > Change runtime type > T4 GPU > Save, then re-run this cell.

!pip install -q --upgrade roboflow kaggle kagglehub 2>/dev/null
print("packages installed")

import torch
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

## Cell 2 — Your API keys

Both keys are typed into a hidden prompt — nothing is written to disk, nothing is displayed,
and no `kaggle.json` legacy credential file is used anywhere in this notebook.

- **Kaggle:** your new API token from kaggle.com/settings/api (it starts with `KGAT_`)
- **Roboflow:** your Private API Key from Roboflow Settings → API Keys

**Tip — set them once instead of every session:** click the 🔑 key icon in Colab's left sidebar
and add two secrets named exactly `KAGGLE_API_TOKEN` and `ROBOFLOW_API_KEY`, with notebook access
switched on. This cell picks them up automatically and won't prompt you at all. Useful because
Colab disconnects if you leave it idle and you'd otherwise retype both keys.

In [ ]:
import os, getpass

def get_secret(name, prompt):
    """Colab secret first (🔑 icon in the sidebar), otherwise a hidden prompt."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            print(f"{name}: loaded from Colab secrets")
            return v.strip()
    except Exception:
        pass
    return getpass.getpass(prompt).strip()

KAGGLE_TOKEN = get_secret("KAGGLE_API_TOKEN",
                          "Paste your Kaggle API token (starts with KGAT_), then press Enter: ")
ROBOFLOW_KEY = get_secret("ROBOFLOW_API_KEY",
                          "Paste your Roboflow Private API Key, then press Enter: ")

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN

# The kaggle package authenticates at import time and then DELETES KAGGLE_API_TOKEN from the
# environment, which breaks any later call. So we keep our own copy and hand a fresh environment
# to every Kaggle call instead of depending on os.environ surviving.
KAGGLE_ENV = {**os.environ, "KAGGLE_API_TOKEN": KAGGLE_TOKEN}

assert KAGGLE_TOKEN, "No Kaggle token entered."
assert ROBOFLOW_KEY, "No Roboflow key entered."
if not KAGGLE_TOKEN.startswith("KGAT_"):
    print("NOTE: Kaggle tokens normally start with 'KGAT_'. Double-check you copied the new API "
          "token (not a legacy key) if the next cell fails.")

print("Kaggle token captured   (%d characters)" % len(KAGGLE_TOKEN))
print("Roboflow key captured   (%d characters)" % len(ROBOFLOW_KEY))

### Cell 2b — Check the Kaggle token actually works

Fails fast with a clear message here rather than 10 minutes into a download.

In [ ]:
import subprocess

try:
    r = subprocess.run(
        ["kaggle", "datasets", "files",
         "praveenkottari/bd3-dataset-for-building-defect-detection", "--page-size", "5"],
        capture_output=True, text=True, env=KAGGLE_ENV, timeout=180)
except FileNotFoundError:
    raise SystemExit("The 'kaggle' command is missing — re-run Cell 1, then run this cell again.")

if r.returncode == 0:
    print("Kaggle authentication OK. Sample of the BD3 dataset files:\n")
    print("\n".join(r.stdout.splitlines()[:8]))
else:
    print("KAGGLE AUTH FAILED — read this before continuing:\n")
    print((r.stderr or r.stdout)[-800:])
    print("\nMost common causes:")
    print("  1. Token copied incompletely — re-copy the whole string from kaggle.com/settings/api")
    print("  2. Token was revoked or regenerated — generate a fresh one and re-run Cell 2")
    print("  3. You haven't opened the BD3 dataset page once while logged in")

## Cell 3 — Download the datasets

This pulls from several public sources. Some may fail (datasets get renamed or made private) —
that's expected and handled. The cell prints exactly what succeeded. As long as we end up with
enough images per class in Cell 5, we are fine.

In [ ]:
import os, shutil, subprocess
RAW = "/content/raw"
os.makedirs(RAW, exist_ok=True)
downloaded = {}
BD3_SLUG = "praveenkottari/bd3-dataset-for-building-defect-detection"

# ---------- 1. BD3 from Kaggle (spalling + peeling, real campus building photos) ----------
# Route A: kagglehub - the modern library, reads KAGGLE_API_TOKEN directly, caches downloads.
try:
    import kagglehub
    bd3_path = kagglehub.dataset_download(BD3_SLUG)
    downloaded["bd3"] = bd3_path
    print("OK   BD3 downloaded via kagglehub ->", bd3_path)
except Exception as e:
    print("kagglehub route failed (%s), trying the CLI..." % str(e)[:120])

    # Route B: the kaggle CLI, given a fresh environment holding the token.
    dest = f"{RAW}/bd3"
    os.makedirs(dest, exist_ok=True)
    last_err = ""
    for args in ([BD3_SLUG], ["-d", BD3_SLUG]):   # positional form, then legacy flag form
        try:
            r = subprocess.run(["kaggle", "datasets", "download", *args,
                                "-p", dest, "--unzip", "-q"],
                               capture_output=True, text=True, env=KAGGLE_ENV, timeout=2400)
        except FileNotFoundError:
            last_err = "the 'kaggle' command is missing - re-run Cell 1"
            break
        if r.returncode == 0:
            downloaded["bd3"] = dest
            print("OK   BD3 downloaded via CLI ->", dest)
            break
        last_err = (r.stderr or r.stdout)[-400:]
    if "bd3" not in downloaded:
        print("FAIL BD3 -", last_err)

# ---------- 2. Roboflow datasets ----------
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_KEY)

# (workspace, project, nickname)
RF_SOURCES = [
    ("stagnant-water",  "stagnant-water",       "water"),
    ("joe-i4soa",       "sep",                  "sep"),
    ("chew-poh-yee",    "internal-wall-defect", "wall"),
    ("cognate-prqm5",   "tile-classification-34erw", "tileA"),
    ("hku-sdauc",       "building-surface",     "tileB"),
]

for ws, proj, nick in RF_SOURCES:
    try:
        p = rf.workspace(ws).project(proj)
        vers = p.versions()
        if not vers:
            print("FAIL", nick, "- no versions"); continue
        v = vers[0]                      # newest version
        d = v.download("coco", location=f"{RAW}/{nick}", overwrite=True)
        downloaded[nick] = f"{RAW}/{nick}"
        print("OK ", nick, "downloaded")
    except Exception as e:
        print("FAIL", nick, "-", str(e)[:160])

print("\nDownloaded sources:", list(downloaded))

## Cell 4 — Look at what we actually got

Before building the dataset we inspect every source so we know which labels exist.
This matters because dataset owners rename classes, and our sorting rules depend on those names.

In [ ]:
import os, json, glob
from collections import Counter

def coco_cats(root):
    out = Counter()
    for jf in glob.glob(os.path.join(root, "**", "_annotations.coco.json"), recursive=True):
        try:
            d = json.load(open(jf))
            names = {c["id"]: c["name"] for c in d.get("categories", [])}
            for a in d.get("annotations", []):
                out[names.get(a["category_id"], "?")] += 1
        except Exception:
            pass
    return out

for nick, path in downloaded.items():
    print("=" * 70)
    print(nick, "->", path)
    if nick == "bd3":
        # BD3 is a folder-per-class dataset; find folders that contain images
        for dirpath, dirnames, filenames in os.walk(path):
            imgs = [f for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png"))]
            if len(imgs) > 20:
                print(f"   {len(imgs):5d}  {os.path.relpath(dirpath, path)}")
    else:
        cats = coco_cats(path)
        for name, n in cats.most_common():
            print(f"   {n:5d}  boxes: {name}")

## Cell 5 — Build the 4-class dataset

Two techniques here:

- **BD3** is already sorted into folders by defect, so we copy whole images.
- The **Roboflow** sets are object-detection data (boxes drawn around defects). We *crop* each box
  with a bit of padding around it. Cropping is deliberate: it gives the model a clean close-up of
  the defect and strips away background that would otherwise let it cheat by recognising which
  dataset a photo came from.

If a class comes out short on images, tell me the counts and I'll add more sources.

In [ ]:
import os, json, glob, shutil, hashlib, random
from PIL import Image
from collections import defaultdict

random.seed(42)
OUT = "/content/dataset_all"
CLASSES = ["spalling", "stagnant_water", "cracked_tiles", "paint_peeling"]
for cl in CLASSES:
    os.makedirs(f"{OUT}/{cl}", exist_ok=True)

MAX_PER_CLASS = 1400
PAD = 0.25         # crop padding around each box
MIN_BOX = 48       # ignore tiny boxes
saved = defaultdict(int)
hashes = set()

def put(img, cls, tag):
    "Save a PIL image into a class folder, skipping duplicates and tiny images."
    if saved[cls] >= MAX_PER_CLASS:
        return False
    if img.width < 64 or img.height < 64:
        return False
    img = img.convert("RGB")
    h = hashlib.md5(img.resize((32, 32)).tobytes()).hexdigest()
    if h in hashes:
        return False
    hashes.add(h)
    img.save(f"{OUT}/{cls}/{tag}_{saved[cls]:05d}.jpg", quality=92)
    saved[cls] += 1
    return True

# ---------- BD3: whole images from the spalling and peeling folders ----------
if "bd3" in downloaded:
    rules = [("spall", "spalling"), ("peel", "paint_peeling")]
    for dirpath, _, filenames in os.walk(downloaded["bd3"]):
        low = os.path.basename(dirpath).lower()
        for key, cls in rules:
            if key in low:
                imgs = [f for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png"))]
                random.shuffle(imgs)
                for f in imgs:
                    try:
                        put(Image.open(os.path.join(dirpath, f)), cls, "bd3")
                    except Exception:
                        pass

# ---------- Roboflow: crop boxes, mapping source label -> our class ----------
# keyword found in the source label  ->  our class
LABEL_RULES = [
    ("tile",     "cracked_tiles"),
    ("crack",    "cracked_tiles"),      # only applied inside tile-focused datasets, see below
    ("spall",    "spalling"),
    ("peel",     "paint_peeling"),
    ("flak",     "paint_peeling"),
    ("water",    "stagnant_water"),
    ("puddle",   "stagnant_water"),
    ("stagnant", "stagnant_water"),
]
# datasets where a bare "crack" label means a cracked tile
TILE_CONTEXT = {"tileA", "tileB"}

def map_label(label, nick):
    l = label.lower()
    for key, cls in LABEL_RULES:
        if key in l:
            if key == "crack" and nick not in TILE_CONTEXT and "tile" not in l:
                continue          # a generic wall crack is not one of our 4 classes
            return cls
    return None

for nick, path in downloaded.items():
    if nick == "bd3":
        continue
    for jf in glob.glob(os.path.join(path, "**", "_annotations.coco.json"), recursive=True):
        folder = os.path.dirname(jf)
        try:
            d = json.load(open(jf))
        except Exception:
            continue
        cats = {c["id"]: c["name"] for c in d.get("categories", [])}
        imgs = {i["id"]: i["file_name"] for i in d.get("images", [])}
        anns = d.get("annotations", [])
        random.shuffle(anns)
        for a in anns:
            cls = map_label(cats.get(a["category_id"], ""), nick)
            if not cls or saved[cls] >= MAX_PER_CLASS:
                continue
            fp = os.path.join(folder, imgs.get(a["image_id"], ""))
            if not os.path.exists(fp):
                continue
            try:
                im = Image.open(fp)
                x, y, w, h = a["bbox"]
                if w < MIN_BOX or h < MIN_BOX:
                    continue
                px, py = w * PAD, h * PAD
                box = (max(0, x - px), max(0, y - py),
                       min(im.width, x + w + px), min(im.height, y + h + py))
                put(im.crop(box), cls, nick)
            except Exception:
                pass

print("\nImages collected per class:")
for cl in CLASSES:
    print(f"  {saved[cl]:5d}  {cl}")
print("\nTOTAL:", sum(saved.values()))

## Cell 6 — Look at the images

Never train on data you haven't looked at. Check that each row genuinely shows that defect.
If a row looks wrong (e.g. `cracked_tiles` full of factory close-ups), stop and tell me.

In [ ]:
import matplotlib.pyplot as plt, random, os
from PIL import Image

fig, axes = plt.subplots(len(CLASSES), 6, figsize=(15, 2.6 * len(CLASSES)))
for r, cl in enumerate(CLASSES):
    fs = os.listdir(f"{OUT}/{cl}")
    random.shuffle(fs)
    for c in range(6):
        ax = axes[r][c]; ax.axis("off")
        if c < len(fs):
            ax.imshow(Image.open(f"{OUT}/{cl}/{fs[c]}").resize((160, 160)))
        if c == 0:
            ax.set_title(f"{cl}  (n={len(fs)})", loc="left", fontsize=11, color="tab:blue")
plt.tight_layout(); plt.show()

## Cell 7 — Split into train / validation / test

- **train (70%)** — the model learns from these
- **validation (15%)** — used during training to pick the best epoch
- **test (15%)** — untouched until the very end; this is what our reported scores come from

Splitting *before* training and never looking at test until the end is what makes the numbers in
our documentation honest.

In [ ]:
import os, random, shutil
random.seed(42)
SPLIT = "/content/dataset"
if os.path.exists(SPLIT):
    shutil.rmtree(SPLIT)

ratios = {"train": 0.70, "val": 0.15, "test": 0.15}
for sp in ratios:
    for cl in CLASSES:
        os.makedirs(f"{SPLIT}/{sp}/{cl}", exist_ok=True)

for cl in CLASSES:
    fs = sorted(os.listdir(f"{OUT}/{cl}"))
    random.shuffle(fs)
    n = len(fs)
    n_tr = int(n * ratios["train"])
    n_va = int(n * ratios["val"])
    parts = {"train": fs[:n_tr], "val": fs[n_tr:n_tr + n_va], "test": fs[n_tr + n_va:]}
    for sp, group in parts.items():
        for f in group:
            shutil.copy(f"{OUT}/{cl}/{f}", f"{SPLIT}/{sp}/{cl}/{f}")

print(f"{'class':<18}{'train':>8}{'val':>8}{'test':>8}")
for cl in CLASSES:
    print(f"{cl:<18}" + "".join(f"{len(os.listdir(f'{SPLIT}/{sp}/{cl}')):>8}" for sp in ratios))

## Cell 8 — Data loaders and augmentation

Augmentation = randomly altering training images (flips, rotations, colour shifts, crops) so the
model sees more variety than we actually have. This is our main defence against the model
memorising individual photos or learning source-specific quirks like lighting.

Validation and test images get no augmentation — only resizing — because we want to measure
performance on clean, realistic inputs.

In [ ]:
import torch, os
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms

IMG = 224
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet stats

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12)),
])
eval_tf = transforms.Compose([
    transforms.Resize(int(IMG * 1.14)),
    transforms.CenterCrop(IMG),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_ds = datasets.ImageFolder(f"{SPLIT}/train", train_tf)
val_ds   = datasets.ImageFolder(f"{SPLIT}/val",   eval_tf)
test_ds  = datasets.ImageFolder(f"{SPLIT}/test",  eval_tf)
CLASS_NAMES = train_ds.classes
print("class order (this order matters for the web app):", CLASS_NAMES)

# balance classes: rarer classes get sampled more often
counts = [0] * len(CLASS_NAMES)
for _, y in train_ds.samples:
    counts[y] += 1
w_per_class = [1.0 / c for c in counts]
sample_w = [w_per_class[y] for _, y in train_ds.samples]
sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

BS = 32
train_dl = DataLoader(train_ds, batch_size=BS, sampler=sampler, num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BS, shuffle=False, num_workers=2)
test_dl  = DataLoader(test_ds,  batch_size=BS, shuffle=False, num_workers=2)
print("train/val/test sizes:", len(train_ds), len(val_ds), len(test_ds))

## Cell 9 — Build the model

**Transfer learning:** ResNet18 has already learned generic visual features (edges, textures,
patterns) from ImageNet. We keep all of that and only replace the final layer with a fresh one
that outputs our 4 classes, then fine-tune the whole thing gently.

The backbone gets a much smaller learning rate than the new head — we want to nudge the existing
knowledge, not wreck it.

In [ ]:
import torch, torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)  # generic ImageNet only
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW([
    {"params": [p for n, p in model.named_parameters() if not n.startswith("fc")], "lr": 3e-5},
    {"params": model.fc.parameters(), "lr": 1e-3},
], weight_decay=1e-4)

EPOCHS = 14
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[3e-4, 3e-3], steps_per_epoch=len(train_dl), epochs=EPOCHS)

print(f"model ready on {device} | {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

## Cell 10 — Train

Watch **val_F1**. It should climb and then flatten. We keep whichever epoch scored highest —
not the last one — because later epochs often start overfitting (memorising training images).

Expect roughly 5–12 minutes on a T4.

In [ ]:
import torch, copy, time
from sklearn.metrics import f1_score

best_f1, best_state, history = 0.0, None, []

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train(); run_loss = 0.0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step(); scheduler.step()
        run_loss += loss.item() * x.size(0)

    model.eval(); preds, gts = [], []
    with torch.no_grad():
        for x, y in val_dl:
            out = model(x.to(device))
            preds += out.argmax(1).cpu().tolist(); gts += y.tolist()

    f1  = f1_score(gts, preds, average="macro")
    acc = sum(int(a == b) for a, b in zip(preds, gts)) / len(gts)
    history.append({"epoch": ep, "loss": run_loss / len(train_ds), "val_acc": acc, "val_f1": f1})
    star = ""
    if f1 > best_f1:
        best_f1, best_state, star = f1, copy.deepcopy(model.state_dict()), "  <-- best so far"
    print(f"epoch {ep:2d}/{EPOCHS} | loss {run_loss/len(train_ds):.4f} | "
          f"val_acc {acc:.4f} | val_F1 {f1:.4f} | {time.time()-t0:.0f}s{star}")

model.load_state_dict(best_state)
print(f"\nBest validation macro-F1: {best_f1:.4f}")

## Cell 11 — Final evaluation on the held-out test set

These are the numbers that go into the documentation report (worth 2% on their own, and they
back up the Detection & Classification section worth 30%).

**How to read the confusion matrix:** each row is the true class, each column is what the model
guessed. The diagonal is correct predictions. Bright cells off the diagonal show which classes
the model confuses with each other.

In [ ]:
import torch, numpy as np, json, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

model.eval(); preds, gts, probs = [], [], []
with torch.no_grad():
    for x, y in test_dl:
        out = torch.softmax(model(x.to(device)), 1)
        probs += out.cpu().tolist()
        preds += out.argmax(1).cpu().tolist()
        gts   += y.tolist()

report = classification_report(gts, preds, target_names=CLASS_NAMES, digits=4)
print(report)

report_d = classification_report(gts, preds, target_names=CLASS_NAMES, output_dict=True)
cm = confusion_matrix(gts, preds)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
plt.xticks(rotation=30, ha="right"); plt.title("InfraPulse — test set confusion matrix")
plt.tight_layout(); plt.savefig("/content/confusion_matrix.png", dpi=150); plt.show()

metrics = {
    "classes": CLASS_NAMES,
    "test_accuracy": report_d["accuracy"],
    "macro_f1": report_d["macro avg"]["f1-score"],
    "per_class": {c: report_d[c] for c in CLASS_NAMES},
    "confusion_matrix": cm.tolist(),
    "val_best_macro_f1": best_f1,
    "history": history,
    "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_ds),
}
json.dump(metrics, open("/content/metrics.json", "w"), indent=2)
print("test accuracy: %.4f | macro F1: %.4f" % (metrics["test_accuracy"], metrics["macro_f1"]))

## Cell 12 — Export the model

Downloads three files:

- `infrapulse_model.pt` — the trained model (this goes into the web app)
- `metrics.json` — the scores (goes into the documentation report)
- `confusion_matrix.png` — the chart (goes into the report and the presentation)

Save all three into your `Descon Mid_prep` folder when they land in your Downloads.

In [ ]:
import torch, json
from google.colab import files

torch.save({
    "arch": "resnet18",
    "state_dict": model.state_dict(),
    "classes": CLASS_NAMES,
    "img_size": IMG,
    "norm_mean": mean,
    "norm_std": std,
    "test_accuracy": metrics["test_accuracy"],
    "macro_f1": metrics["macro_f1"],
}, "/content/infrapulse_model.pt")

import os
print("model size: %.1f MB" % (os.path.getsize("/content/infrapulse_model.pt") / 1e6))

files.download("/content/infrapulse_model.pt")
files.download("/content/metrics.json")
files.download("/content/confusion_matrix.png")